In [ ]:
# models
from fastapi import FastAPI
from pydantic import BaseModel

# websites
import requests

import joblib
import numpy as pn

## Zadanie 0
Należy utworzyć bardzo prostą stronę internetową zawierającą tytuł, kilka nagłówków i paragrafów. Jeden paragraf ma zawierać hiperłącze do strony MiMUW. Utworzoną stronę proszę zapisać w pliku i podejrzeć w przeglądarce z wykorzystaniem "Zbadaj" (zazwyczaj F12).
Umieścić tę stronę w katalogu public_html. Za pomocą BeutifulSoap przetwórz tę stronę - np. wypisz wszystkie linki do innych stron.


public html, to wrzucić, i dostępne dla wszystkich (ale nie zapis), wtedy każdy może swoją stronę postawić. 

## Zadanie 1

Wybierz jeden lub kilka genów/białek, np. **BRCA1**, **TP53**, **EGFR**.

Napisz funkcję w Pythonie, która:

1. przyjmuje jako argument identyfikator UniProt (np. `"P04637"` dla TP53),
2. łączy się z API UniProt (`https://rest.uniprot.org/uniprotkb/{id}.json`),
3. pobiera dane w formacie JSON,
4. zwraca słownik z następującymi informacjami:
   - `protein_name` – nazwa białka,
   - `gene_name` – nazwa genu,
   - `organism` – nazwa organizmu,
   - `sequence` – sekwencja aminokwasowa białka.

Zadbaj, aby funkcja była odporna na brak niektórych pól w odpowiedzi API (nie wszystkie białka muszą mieć wszystkie dane).

Przetestuj funkcję na wybranych białkach i wyświetl wyniki w czytelnej formie.

**Wskazówka:**  
Do pobierania danych użyj biblioteki `requests`. Zwróć uwagę, że struktura JSON w UniProt może się różnić w zależności od białka, dlatego warto korzystać z metod typu `.get()` lub sprawdzać obecność kluczy.


co my tutaj właściwie dostajemy, to dotyczy danych wszystkich na stronie, czy co my tutaj możemy znaleźć 

In [ ]:
from urllib import response


def getName(name, data):
    # finding dict 
    search_name = data[name] if name in data.keys() else None
    # finding condition
    if search_name is not None:
        if type(search_name) is list:
            search_name = search_name.pop(0)

        search_name =  search_name[list(search_name.keys())[0]]
    return search_name

def uniprot_manager(id: str)->dict:
    '''We download data connected to `id` experiment`'''
    # connecting to Uniprot
    link = f'https://www.uniprot.org/uniprotkb/{id}.json'
    response = requests.get(link)
    # checking status
    if response.status_code != 200:
        print('Connection unsuccessful')
    else:
        print('Connection successful')
    # converting response to json
    data = response.json() 

    # protein_name (i do not assume condition - which part of info)
    protein_name = getName('proteinDescription', data)
    # gene_name (i do not assume condition - which part of info)
    gene_name = getName('genes', data)
    #organism 
    organism = getName('organism', data)
    # organism = {key: data['organism'][key]  for key in list(data['organism'].keys())[:2]} if 'genes' in data.keys() else None

    # sequence
    sequence = data['sequence']['value'] if 'sequence' in data.keys() else None




    return protein_name, gene_name, organism, sequence
    
uniprot_manager('P04637')

## Zadanie 2

Napisz skrypt w Pythonie, który wykorzysta REST API Ensembl VEP, aby automatycznie ocenić efekt podanych wariantów DNA (np. czy jest to `missense_variant`, `stop_gained`, `3_prime_UTR_variant` itp.) i przedstawić wyniki w czytelnej formie.

1. Przygotuj listę wariantów w formacie `"<chrom> <pos> . <ref> <alt>"`, np. `"17 7676154 . G A"`.
2. Wyślij zapytanie `POST` do `https://rest.ensembl.org/vep/human/region/` z JSON-em zawierającym te warianty.
3. Odbierz odpowiedź (JSON) i dla każdego wariantu wypisz co najmniej:
   - nazwę genu (`gene_symbol`),
   - najbardziej prawdopodobny skutek (`most_severe_consequence`).
4. (Opcjonalnie) Zapisz wyniki do pliku CSV.

In [ ]:
variants = [
    "17 7676154 . G A",    # TP53 — znana mutacja (prawdopodobnie missense)
    "17 43071077 . A G",   # BRCA1
    "7 55249071 . G A"     # EGFR
] # i dodaj własne

## Zadanie 3

- Wybierz jeden lub kilka autobusów (np. po numerze linii).
- Pobierz ich pozycje **trzy razy w odstępie jednej minuty**, korzystając z asynchronicznego API ZTM.
- Umieść te pozycje na mapie przy użyciu biblioteki `folium`.

**Wskazówki:**

- Użyj bibliotek `asyncio` i `aiohttp` do wykonywania wielu żądań jednocześnie.
- Zadbaj o obsługę błędów sieciowych i brakujących danych (nie wszystkie autobusy raportują pozycję w danym momencie).
- Zastanów się, jak agregować dane i wizualnie połączyć punkty trasą na mapie.

**Rozszerzenie**

Wypisz wszyskie autobusy danej linii. Umożliw wybranie do analizy dowolnego z nich. Aby uniknąć analizowanie akurat stojącego na pętli.

trzeba ułozyć konto, login i hasło 
losowe dane użytkownika i hasło

In [ ]:
from pathlib import Path
import time 


In [ ]:
# Wersja z wyborem id_autobusu; API zwraca wszystkie autobusy; Można dodać oprócz ograniczenia numeru id_autobusu, 
# ograniczenie numeru linii już dla pobranych wyników;

In [ ]:
# starsza wersja wskazała autobus w zajezdni

## Zadanie 4
Odtwórz z wykładu.

#### Krok 1. Trenowanie i zapis modelu

- Wczytaj zbiór danych Iris z `sklearn.datasets`.
- Wytrenuj model klasyfikacyjny Random Forest na całym zbiorze lub podzielonym na zestaw treningowy i testowy.
- Zapisz wytrenowany model do pliku `model.pkl`.
  
Możesz zastosować inne metody / użyć modelu z zajęć z zeszłego tygodnia.

#### Krok 2. Utworzenie prostego API do predykcji

- Utwórz serwer FastAPI.
- Dodaj endpoint `/predict`, który przyjmuje dane wejściowe w formacie JSON (cechy kwiatu).
- Endpoint powinien wczytać model z `model.pkl` i zwrócić predykcję gatunku kwiatu.

#### Krok 3. Testowanie API lokalnie

- Uruchom serwer FastAPI i przetestuj endpoint `/predict` z przykładowymi danymi.
- Sprawdź poprawność odpowiedzi.

---

#### Wskazówki

- Do trenowania modelu użyj `sklearn.ensemble.RandomForestClassifier`.
- Do zapisu i odczytu modelu możesz wykorzystać bibliotekę `joblib` (`joblib.dump()` i `joblib.load()`).
- FastAPI pozwala na szybkie tworzenie serwerów REST; endpoint można zdefiniować jako funkcję z dekoratorem `@app.post("/predict")`.
- Format danych JSON dla Iris: słownik z kluczami odpowiadającymi cechom kwiatu (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`).
- Do testów lokalnych można użyć np. `curl` lub `requests` w Pythonie, wysyłając dane JSON do endpointu `/predict`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from pydantic import BaseModel
from sklearn.model_selection import train_test_split 
import joblib
import numpy as np 
data_path = 'trained_models'
model_path = data_path + '/random_forest_100'

# trenowanie modelu 
data = load_iris()
X, y =data['data'], data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# eksportowanie modelu 
joblib.dump(model, model_path)
#TODO - porównać dump vs pickle 

# ładowanie modelu 
model_loaded = joblib.load(model_path)
app = FastAPI(title="My API")


# model danych wejściowych 
#TODO - sprawdzić czym jest BaseModel i w jaki sposób jest potrzebny do załadowania tych danych 
class IrisFeatures(BaseModel):
    #TODO - co jak byśmy mieli bardzo dużo cech ??? 
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

# Stawianie aplikacji 
@app.post("/predict") # tutaj dajemy sobie podstronę 
def predykcja_gatunków(features: IrisFeatures):
    # TODO - na czym polega ta konstrukcja 
    data = np.array([[features.sepal_length,
                      features.sepal_width,
                      features.petal_length,
                      features.petal_width]])
    pred = model.predict(data)[0]
    species = target_names[pred]
    return {"Predykcja": species}
    

Jak to uruchomic xdd 